In [ ]:
# ======================================================================================
# ROBUST ECG IMAGE -> 12-LEAD PSEUDO-WAVEFORM EXTRACTION
# VERSION 2: GRID REMOVAL + CONTINUITY-CONSTRAINED TRACE TRACKING
# ======================================================================================
#
# OUTPUTS:
#   digitization_qc_v2/
#       GRID_<ECG>.png                 <- verifies the 12 crop locations
#       QC2_<ECG>.png                  <- source crop + detected trace + waveform
#       digitization_qc_metrics.csv
#       X_gaze_v2.npy                  <- final (10,12,1000,1) array
#
# IMPORTANT:
# Do NOT retrain the model yet.
# First inspect QC2_*.png.
#
# ======================================================================================

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageOps

from scipy.ndimage import (
    binary_opening,
    binary_dilation,
    distance_transform_edt,
    median_filter,
    gaussian_filter1d,
)

from scipy.signal import savgol_filter


# ======================================================================================
# 1. PATHS
# ======================================================================================

EYE_ROOT = (
    "eye_tracking/"
    "eye-tracking-dataset-for-the-12-lead-electrocardiogram-"
    "interpretation-of-medical-practitioners-and-students-1.0.0"
)

OUT_DIR = "digitization_qc_v2"
os.makedirs(OUT_DIR, exist_ok=True)


# ======================================================================================
# 2. ECG DEFINITIONS
# ======================================================================================

ECG_KEYS = [
    "NSR",
    "AFib",
    "Hyperkalemia",
    "AFlutter",
    "VTach",
    "WPW",
    "VentPaced",
    "LBBB",
    "STEMI",
    "CompleteHeartBlock",
]

IMAGE_FILES = {
    "NSR": "Normal_Sinus_Rhythm.jpg",
    "AFib": "Atrial_Fibrillation.png",
    "Hyperkalemia": "Hyperkalemia.jpg",
    "AFlutter": "Atrial_Flutter.jpg",
    "VTach": "Ventricular_Tachycardia.jpg",
    "WPW": "Wolf_Parkinson_White_Syndrome.jpg",
    "VentPaced": "Ventricular_Paced_Rhythm.jpg",
    "LBBB": "Left_Bundle_Branch_Block.jpg",
    "STEMI": "STEMI.png",
    "CompleteHeartBlock": "Complete_Heart_Block.jpg",
}


# Physical layout on the source ECG images
DISPLAY_LEADS = [
    ["I",   "aVR", "V1", "V4"],
    ["II",  "aVL", "V2", "V5"],
    ["III", "aVF", "V3", "V6"],
]

# Correct order expected by the pretrained waveform model
STANDARD_LEADS = [
    "I", "II", "III",
    "aVR", "aVL", "aVF",
    "V1", "V2", "V3",
    "V4", "V5", "V6",
]


# ======================================================================================
# 3. GRID SETTINGS
# ======================================================================================
#
# These define the overall 3 x 4 ECG area.
#
# If the GRID_*.png images show that the rectangles are misplaced,
# these are the ONLY numbers we should modify.
#
# format = (left, top, right, bottom) as fractions of whole image
# ======================================================================================

DEFAULT_GRID_BOUNDS = (0.015, 0.04, 0.985, 0.96)

# Override individual images here ONLY if visually required.
GRID_BOUNDS_BY_ECG = {
    # Example:
    # "NSR": (0.02, 0.05, 0.98, 0.94),
}


# Padding inside each lead box.
# This helps avoid lead labels / borders / neighbouring traces.
CELL_PAD_LEFT   = 0.045
CELL_PAD_RIGHT  = 0.045
CELL_PAD_TOP    = 0.12
CELL_PAD_BOTTOM = 0.10


# ======================================================================================
# 4. TRACE EXTRACTION SETTINGS
# ======================================================================================

WORK_HEIGHT = 150
WORK_WIDTH = 750

OUTPUT_SAMPLES = 1000

# Maximum vertical movement between adjacent columns
MAX_VERTICAL_JUMP = 10

# Penalty for jumping vertically.
SMOOTHNESS_PENALTY = 0.17

# Grid-line morphological lengths
HORIZONTAL_LINE_LENGTH = 38
VERTICAL_LINE_LENGTH = 28

# Threshold percentile for dark ECG ink
INK_PERCENTILE = 78


# ======================================================================================
# 5. HELPERS
# ======================================================================================

def find_image(filename):

    preferred = os.path.join(
        EYE_ROOT,
        "ECGs",
        "ECG_Images",
        filename
    )

    if os.path.exists(preferred):
        return preferred

    matches = glob.glob(
        os.path.join(
            EYE_ROOT,
            "**",
            filename
        ),
        recursive=True
    )

    if len(matches) == 0:
        raise FileNotFoundError(filename)

    return matches[0]


def resize_gray(array, width, height):

    arr = np.asarray(array)

    if arr.dtype != np.uint8:

        if arr.max() <= 1.0:
            arr = arr * 255.0

        arr = np.clip(
            arr,
            0,
            255
        ).astype(np.uint8)

    img = Image.fromarray(arr)

    try:
        method = Image.Resampling.BILINEAR
    except AttributeError:
        method = Image.BILINEAR

    img = img.resize(
        (width, height),
        resample=method
    )

    return (
        np.asarray(img)
        .astype(np.float32)
        / 255.0
    )


def robust_normalise_image(gray):

    gray = np.asarray(
        gray,
        dtype=np.float32
    )

    lo = np.percentile(
        gray,
        1
    )

    hi = np.percentile(
        gray,
        99
    )

    if hi <= lo:
        return gray

    gray = (
        gray - lo
    ) / (
        hi - lo
    )

    return np.clip(
        gray,
        0,
        1
    )


# ======================================================================================
# 6. REMOVE GRID / LONG STRAIGHT STRUCTURES
# ======================================================================================

def clean_ecg_ink(gray):

    gray = robust_normalise_image(
        gray
    )

    darkness = 1.0 - gray

    # Slight blur removes isolated JPEG noise
    darkness_smooth = gaussian_filter1d(
        darkness,
        sigma=0.6,
        axis=1
    )

    threshold = np.percentile(
        darkness_smooth,
        INK_PERCENTILE
    )

    threshold = max(
        threshold,
        0.08
    )

    ink = (
        darkness_smooth
        >
        threshold
    )

    # ------------------------------------------------------------
    # Detect LONG horizontal structures
    # ECG traces should not remain horizontal across 38+ pixels
    # nearly continuously.
    # ------------------------------------------------------------

    horizontal_structure = np.ones(
        (
            1,
            HORIZONTAL_LINE_LENGTH
        ),
        dtype=bool
    )

    horizontal_lines = binary_opening(
        ink,
        structure=horizontal_structure
    )

    horizontal_lines = binary_dilation(
        horizontal_lines,
        structure=np.ones(
            (3, 1),
            dtype=bool
        )
    )

    # ------------------------------------------------------------
    # Detect LONG vertical grid/border structures
    # ------------------------------------------------------------

    vertical_structure = np.ones(
        (
            VERTICAL_LINE_LENGTH,
            1
        ),
        dtype=bool
    )

    vertical_lines = binary_opening(
        ink,
        structure=vertical_structure
    )

    vertical_lines = binary_dilation(
        vertical_lines,
        structure=np.ones(
            (1, 3),
            dtype=bool
        )
    )

    cleaned = (
        ink
        &
        ~horizontal_lines
        &
        ~vertical_lines
    )

    return (
        darkness,
        ink,
        cleaned
    )


# ======================================================================================
# 7. DYNAMIC-PROGRAMMING TRACE TRACKER
# ======================================================================================

def track_continuous_trace(
    cleaned_ink,
    darkness
):

    H, W = cleaned_ink.shape

    # ------------------------------------------------------------
    # Cost of each pixel:
    # distance to nearest remaining dark ECG pixel.
    #
    # A true ECG path should remain close to dark ink.
    # ------------------------------------------------------------

    distance = distance_transform_edt(
        ~cleaned_ink
    ).astype(np.float32)

    distance = np.minimum(
        distance,
        15
    )

    distance /= 15.0

    # Reward pixels that are still visually dark even if thresholding
    # removed them.
    dark_reward = (
        darkness
        /
        (
            darkness.max()
            +
            1e-8
        )
    )

    observation_cost = (
        distance
        -
        0.32 * dark_reward
    )

    # Avoid image edges
    edge_penalty = np.zeros(
        H,
        dtype=np.float32
    )

    edge_zone = max(
        3,
        H // 20
    )

    edge_penalty[
        :edge_zone
    ] = 1.2

    edge_penalty[
        -edge_zone:
    ] = 1.2

    observation_cost += edge_penalty[:, None]


    # ------------------------------------------------------------
    # Dynamic programming
    # ------------------------------------------------------------

    J = MAX_VERTICAL_JUMP

    jumps = np.arange(
        -J,
        J + 1
    )

    previous = (
        observation_cost[
            :,
            0
        ].copy()
    )

    back_pointer = np.zeros(
        (
            H,
            W
        ),
        dtype=np.int16
    )

    INF = 1e12

    y = np.arange(
        H
    )

    for x in range(
        1,
        W
    ):

        candidate_costs = np.full(
            (
                len(jumps),
                H
            ),
            INF,
            dtype=np.float32
        )

        for k, jump in enumerate(
            jumps
        ):

            source_y = (
                y - jump
            )

            valid = (
                (source_y >= 0)
                &
                (source_y < H)
            )

            candidate_costs[
                k,
                valid
            ] = (
                previous[
                    source_y[
                        valid
                    ]
                ]
                +
                SMOOTHNESS_PENALTY
                *
                abs(
                    jump
                )
            )

        best_jump_index = np.argmin(
            candidate_costs,
            axis=0
        )

        best_previous_cost = candidate_costs[
            best_jump_index,
            y
        ]

        current = (
            observation_cost[
                :,
                x
            ]
            +
            best_previous_cost
        )

        best_jumps = jumps[
            best_jump_index
        ]

        back_pointer[
            :,
            x
        ] = (
            y
            -
            best_jumps
        ).astype(
            np.int16
        )

        previous = current


    # ------------------------------------------------------------
    # Backtrack optimal path
    # ------------------------------------------------------------

    path = np.zeros(
        W,
        dtype=np.float32
    )

    path[
        -1
    ] = np.argmin(
        previous
    )

    for x in range(
        W - 1,
        0,
        -1
    ):

        current_y = int(
            round(
                path[
                    x
                ]
            )
        )

        path[
            x - 1
        ] = back_pointer[
            current_y,
            x
        ]


    # ------------------------------------------------------------
    # Remove isolated tracker jitter
    # ------------------------------------------------------------

    path = median_filter(
        path,
        size=5
    )

    try:

        path = savgol_filter(
            path,
            window_length=15,
            polyorder=2,
            mode="interp"
        )

    except Exception:
        pass


    # Image y increases DOWN.
    # ECG amplitude convention increases UP.
    path = -path


    # Remove offset
    path -= np.median(
        path
    )


    return path


# ======================================================================================
# 8. RESAMPLE TRACKED PATH TO 1000 POINTS
# ======================================================================================

def path_to_model_signal(
    path
):

    path = np.asarray(
        path,
        dtype=np.float32
    )

    old_x = np.linspace(
        0,
        1,
        len(
            path
        )
    )

    new_x = np.linspace(
        0,
        1,
        OUTPUT_SAMPLES
    )

    signal = np.interp(
        new_x,
        old_x,
        path
    )

    signal -= np.mean(
        signal
    )

    signal /= (
        np.std(
            signal
        )
        +
        1e-8
    )

    return signal.astype(
        np.float32
    )


# ======================================================================================
# 9. QC METRICS
# ======================================================================================

def trace_qc_metrics(
    raw_path,
    cleaned_ink
):

    H, W = cleaned_ink.shape

    y_path = np.clip(
        np.round(
            -raw_path
            +
            np.median(
                -raw_path
            )
        ),
        0,
        H - 1
    )

    # More reliable:
    # normalise path independently for structural metrics
    p = raw_path.copy()

    jumps = np.abs(
        np.diff(
            p
        )
    )

    # fraction of abnormally abrupt jumps
    jump_fraction = np.mean(
        jumps > 6
    )

    # median jump
    median_jump = np.median(
        jumps
    )

    # smoothness index
    roughness = np.mean(
        np.abs(
            np.diff(
                p,
                n=2
            )
        )
    )

    return {
        "median_vertical_step":
            float(
                median_jump
            ),

        "large_jump_fraction":
            float(
                jump_fraction
            ),

        "second_difference_roughness":
            float(
                roughness
            )
    }


# ======================================================================================
# 10. EXTRACT CELLS
# ======================================================================================

def get_grid_bounds(
    img,
    ecg_key
):

    H, W = img.shape

    bounds = GRID_BOUNDS_BY_ECG.get(
        ecg_key,
        DEFAULT_GRID_BOUNDS
    )

    left, top, right, bottom = bounds

    x0 = int(
        left * W
    )

    x1 = int(
        right * W
    )

    y0 = int(
        top * H
    )

    y1 = int(
        bottom * H
    )

    return (
        x0,
        y0,
        x1,
        y1
    )


def get_cell(
    img,
    grid_bounds,
    row,
    col
):

    gx0, gy0, gx1, gy1 = (
        grid_bounds
    )

    GW = (
        gx1 - gx0
    )

    GH = (
        gy1 - gy0
    )

    x0 = (
        gx0
        +
        int(
            col
            *
            GW
            /
            4
        )
    )

    x1 = (
        gx0
        +
        int(
            (
                col + 1
            )
            *
            GW
            /
            4
        )
    )

    y0 = (
        gy0
        +
        int(
            row
            *
            GH
            /
            3
        )
    )

    y1 = (
        gy0
        +
        int(
            (
                row + 1
            )
            *
            GH
            /
            3
        )
    )

    width = (
        x1 - x0
    )

    height = (
        y1 - y0
    )

    x0 += int(
        CELL_PAD_LEFT
        *
        width
    )

    x1 -= int(
        CELL_PAD_RIGHT
        *
        width
    )

    y0 += int(
        CELL_PAD_TOP
        *
        height
    )

    y1 -= int(
        CELL_PAD_BOTTOM
        *
        height
    )

    return (
        img[
            y0:y1,
            x0:x1
        ],
        (
            x0,
            y0,
            x1,
            y1
        )
    )


# ======================================================================================
# 11. EXTRACT ONE ECG
# ======================================================================================

def extract_one_ecg(
    ecg_key,
    image_path
):

    img = (
        np.asarray(
            Image.open(
                image_path
            ).convert(
                "L"
            )
        )
        .astype(
            np.float32
        )
        /
        255.0
    )

    bounds = get_grid_bounds(
        img,
        ecg_key
    )

    lead_signals = {}

    lead_debug = {}

    metrics = []


    for row in range(
        3
    ):

        for col in range(
            4
        ):

            lead = DISPLAY_LEADS[
                row
            ][
                col
            ]

            cell_original, box = (
                get_cell(
                    img,
                    bounds,
                    row,
                    col
                )
            )

            cell = resize_gray(
                cell_original,
                WORK_WIDTH,
                WORK_HEIGHT
            )

            darkness, raw_ink, cleaned_ink = (
                clean_ecg_ink(
                    cell
                )
            )

            raw_path = (
                track_continuous_trace(
                    cleaned_ink,
                    darkness
                )
            )

            signal = (
                path_to_model_signal(
                    raw_path
                )
            )

            lead_signals[
                lead
            ] = signal

            lead_debug[
                lead
            ] = {
                "cell":
                    cell,

                "darkness":
                    darkness,

                "raw_ink":
                    raw_ink,

                "cleaned_ink":
                    cleaned_ink,

                "raw_path":
                    raw_path,

                "box":
                    box,
            }

            q = trace_qc_metrics(
                raw_path,
                cleaned_ink
            )

            q.update(
                {
                    "ECG":
                        ecg_key,

                    "lead":
                        lead,
                }
            )

            metrics.append(
                q
            )


    X = np.stack(
        [
            lead_signals[
                lead
            ]
            for lead
            in STANDARD_LEADS
        ],
        axis=0
    )

    return (
        X[
            ...,
            None
        ],
        lead_debug,
        metrics,
        img,
        bounds
    )


# ======================================================================================
# 12. GRID QC IMAGE
# ======================================================================================

def save_grid_qc(
    ecg_key,
    img,
    bounds,
    debug
):

    plt.figure(
        figsize=(
            16,
            9
        )
    )

    plt.imshow(
        img,
        cmap="gray"
    )

    colors = plt.cm.tab20(
        np.linspace(
            0,
            1,
            12
        )
    )

    for i, lead in enumerate(
        STANDARD_LEADS
    ):

        x0, y0, x1, y1 = (
            debug[
                lead
            ][
                "box"
            ]
        )

        rectangle = (
            plt.Rectangle(
                (
                    x0,
                    y0
                ),
                x1 - x0,
                y1 - y0,
                fill=False,
                linewidth=2,
                color=colors[
                    i
                ]
            )
        )

        plt.gca().add_patch(
            rectangle
        )

        plt.text(
            x0 + 5,
            y0 + 18,
            lead,
            fontsize=10,
            color=colors[
                i
            ],
            backgroundcolor="white"
        )

    plt.title(
        f"{ecg_key}: lead crop locations"
    )

    plt.axis(
        "off"
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUT_DIR,
            f"GRID_{ecg_key}.png"
        ),
        dpi=180
    )

    plt.close()


# ======================================================================================
# 13. SOURCE-CROP + TRACE-OVERLAY QC
# ======================================================================================

def save_trace_qc(
    ecg_key,
    debug,
    X
):

    fig = plt.figure(
        figsize=(
            18,
            22
        )
    )

    outer = fig.add_gridspec(
        12,
        2,
        width_ratios=[
            1,
            1
        ],
        hspace=0.55
    )

    for i, lead in enumerate(
        STANDARD_LEADS
    ):

        info = debug[
            lead
        ]

        cell = info[
            "cell"
        ]

        path = info[
            "raw_path"
        ]


        # ---------------------------------------------------------
        # LEFT:
        # actual image crop + red detected path
        # ---------------------------------------------------------

        ax1 = fig.add_subplot(
            outer[
                i,
                0
            ]
        )

        ax1.imshow(
            cell,
            cmap="gray",
            aspect="auto",
            origin="upper"
        )

        # Convert inverted path back to image coordinates.
        p = (
            -path
        )

        # Recover vertical offset for visualisation
        p = (
            p
            -
            np.median(
                p
            )
            +
            cell.shape[
                0
            ]
            /
            2
        )

        p = np.clip(
            p,
            0,
            cell.shape[
                0
            ]
            -
            1
        )

        ax1.plot(
            np.arange(
                len(
                    p
                )
            ),
            p,
            linewidth=1.3,
            color="red"
        )

        ax1.set_title(
            f"{lead}: source crop + tracked path"
        )

        ax1.set_xticks([])
        ax1.set_yticks([])


        # ---------------------------------------------------------
        # RIGHT:
        # extracted waveform
        # ---------------------------------------------------------

        ax2 = fig.add_subplot(
            outer[
                i,
                1
            ]
        )

        ax2.plot(
            X[
                i,
                :,
                0
            ],
            linewidth=1
        )

        ax2.axhline(
            0,
            linewidth=0.5,
            alpha=0.4
        )

        ax2.set_title(
            f"{lead}: extracted 1000-sample pseudo-waveform"
        )

        ax2.set_xticks([])
        ax2.set_yticks([])


    fig.suptitle(
        f"{ecg_key} — ECG digitisation QC",
        fontsize=18,
        y=0.995
    )

    fig.tight_layout()

    fig.savefig(
        os.path.join(
            OUT_DIR,
            f"QC2_{ecg_key}.png"
        ),
        dpi=170,
        bbox_inches="tight"
    )

    plt.close(
        fig
    )


# ======================================================================================
# 14. RUN ALL TEN ECGs
# ======================================================================================

all_X = []

all_metrics = []

print(
    "\n" + "=" * 90
)

print(
    "ROBUST ECG DIGITISATION"
)

print(
    "=" * 90
)


for ecg_key in ECG_KEYS:

    path = find_image(
        IMAGE_FILES[
            ecg_key
        ]
    )

    print(
        "\nProcessing:",
        ecg_key
    )

    print(
        "Image:",
        path
    )

    X, debug, metrics, img, bounds = (
        extract_one_ecg(
            ecg_key,
            path
        )
    )

    print(
        "Shape:",
        X.shape
    )

    all_X.append(
        X
    )

    all_metrics.extend(
        metrics
    )

    save_grid_qc(
        ecg_key,
        img,
        bounds,
        debug
    )

    save_trace_qc(
        ecg_key,
        debug,
        X
    )


X_gaze_v2 = np.stack(
    all_X
).astype(
    np.float32
)

print(
    "\nFINAL ARRAY:",
    X_gaze_v2.shape
)

assert (
    X_gaze_v2.shape
    ==
    (
        10,
        12,
        1000,
        1
    )
)


# ======================================================================================
# 15. SAVE ARRAY + METRICS
# ======================================================================================

np.save(
    os.path.join(
        OUT_DIR,
        "X_gaze_v2.npy"
    ),
    X_gaze_v2
)


metrics_df = pd.DataFrame(
    all_metrics
)

metrics_df.to_csv(
    os.path.join(
        OUT_DIR,
        "digitization_qc_metrics.csv"
    ),
    index=False
)


print(
    "\nQC METRICS SUMMARY"
)

print(
    metrics_df[
        [
            "median_vertical_step",
            "large_jump_fraction",
            "second_difference_roughness"
        ]
    ]
    .describe()
    .to_string()
)


# ======================================================================================
# 16. FLAG POTENTIALLY BAD LEADS
# ======================================================================================

rough_threshold = (
    metrics_df[
        "second_difference_roughness"
    ]
    .quantile(
        0.90
    )
)

jump_threshold = max(
    0.05,
    metrics_df[
        "large_jump_fraction"
    ]
    .quantile(
        0.90
    )
)


metrics_df[
    "automatic_QC_flag"
] = (
    (
        metrics_df[
            "second_difference_roughness"
        ]
        >
        rough_threshold
    )
    |
    (
        metrics_df[
            "large_jump_fraction"
        ]
        >
        jump_threshold
    )
)


metrics_df.to_csv(
    os.path.join(
        OUT_DIR,
        "digitization_qc_metrics.csv"
    ),
    index=False
)


print(
    "\nAUTOMATICALLY FLAGGED LEADS"
)

flagged = metrics_df[
    metrics_df[
        "automatic_QC_flag"
    ]
]

if len(
    flagged
):

    print(
        flagged[
            [
                "ECG",
                "lead",
                "median_vertical_step",
                "large_jump_fraction",
                "second_difference_roughness"
            ]
        ]
        .to_string(
            index=False
        )
    )

else:

    print(
        "None automatically flagged."
    )


# ======================================================================================
# 17. FINAL INSTRUCTIONS
# ======================================================================================

print(
    "\n" + "=" * 90
)

print(
    "DIGITISATION COMPLETE"
)

print(
    "=" * 90
)

print(
    f"""
Saved to:

    {OUT_DIR}/

Please inspect:

    GRID_<ECG>.png
        → Are the 12 rectangles actually over the correct leads?

    QC2_<ECG>.png
        → LEFT: source ECG crop with RED tracked waveform
        → RIGHT: reconstructed 1000-point signal

The red line should visibly follow the actual ECG trace.

Do NOT retrain yet.

If these pass visual QC, the final clean experiment only needs:

    X_gaze = np.load(
        "{OUT_DIR}/X_gaze_v2.npy"
    )

instead of the previous image reconstruction step.

Everything else in the clean training/evaluation experiment remains unchanged.
"""
)